In [ ]:
from dep_tools.searchers import PystacSearcher
from dep_tools.loaders import OdcLoader
from src.utils import SDBProcessor, S2_BANDS

from dep_tools.grids import PACIFIC_GRID_10
import joblib

In [ ]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [ ]:
catalog = "https://earth-search.aws.element84.com/v1"
collection = "sentinel-2-l2a"

# tile_id = (130, 12)  # Tiny island
tile_id = (104, 23)  # Another noisy island
# tile_id = (66, 22)  # noise near Eastern Vanua Levu. Still BAD
# tile_id = (48, 18)  # NW of New Caledonia
# tile_id = (64, 20)  # Suva

geobox = PACIFIC_GRID_10.tile_geobox(tile_id)
datetime = "2025-01"

model = joblib.load("models/2025_04_17_rf.joblib")

searcher = PystacSearcher(
    catalog=catalog,
    collections=[collection],
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": 50}},
)

loader = OdcLoader(
    bands=S2_BANDS,
    chunks={"x": 3201, "y": 3201},
    groupby="solar_day",
    fail_on_error=False,
)

processor = SDBProcessor(
    model=model,
    model_tides=True,
    parallelism=8
)

In [4]:
from dep_tools.namers import S3ItemPath

itempath = S3ItemPath(
    bucket="example-bucket",
    sensor="s2",
    dataset_id="sdb",
    version="9.9.9",
    time="2024-07/2024-12",
)

In [ ]:
items = searcher.search(geobox)

print(f"Found {len(items)} items")

In [ ]:
data = loader.load(items, geobox)

data

In [ ]:
results = processor.process(data)

results

In [ ]:
results.pc_deep.plot.hist(bins=11)

In [ ]:
import folium
from ipyleaflet import basemaps

masked = results.where(results.pc_deep < 0.7)  # ||  # Maybe use 0.5
masked_two = results.where(results.pc_pred > 0.1)

m = folium.Map(tiles=basemaps.Esri.WorldImagery)
m.fit_bounds(results.odc.map_bounds())

for var in results.data_vars:
    cmap = "Blues_r" if var in ("mean", "median") else "viridis"
    args = {
        "cmap": cmap,
    }
    if "pc" in var:
        args["vmin"] = 0
        args["vmax"] = 1
    results[var].odc.add_to(m, name=var, **args)

masked["mean"].odc.add_to(m, name="mean masked", cmap="Blues_r")
masked["median"].odc.add_to(m, name="median masked", cmap="Blues_r")

masked_two["mean"].odc.add_to(m, name="mean masked two", cmap="Blues_r")
masked_two["median"].odc.add_to(m, name="median masked two", cmap="Blues_r")

# Add a layer control to the map
folium.LayerControl().add_to(m)

m

In [ ]:
# results["pc_deep"].odc.write_cog("pc_deep_vanua.tif", overwrite=True)